### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 
    Tracking agent behavior with logging, analytics, and debugging.
    Transforming prompts, tool selection, and output formatting.
    Adding retries, fallbacks, and early termination logic.
    Applying rate limits, guardrails, and Pll detection.


In [13]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
    Long Running conversations that exceed context windows.

    Multi-turn dialogues with extensive history.

    Applications where preserving full conversation context matters.

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [15]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [16]:
# Alternative test data
questions = [
    "What is 2 + 2?",
    "What is 10*5",
    "What is 100/4",
    "What is 15-7?",
    "What is 3*3",
    "What is 4*4",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2 + 2?', additional_kwargs={}, response_metadata={}, id='44bac9c2-b4c1-4e4f-95e4-4fca409839af'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a61d6793b9', 'id': 'chatcmpl-Ef0e05ILKuqNkz74jits4LUCZZC2W', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19182-cafe-7f62-b260-8cd802af5934-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'tot

# token Size

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
        model = "gpt-4o-mini",
        tools = [search_hotels],
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger = ("tokens", 550),
                keep=("tokens", 200)
            )
        ]
    )

config = {"configurable": {"thread_id": "test-1"}}

    # Token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4    

In [18]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~153 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='69a930da-3377-472f-9d91-625eaf3eb221'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9b01c99668', 'id': 'chatcmpl-Ef0e88lxJvdJLmTV3hQ718VylWn44', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19182-eff9-7600-9083-e0e4b9da7075-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_f9tv7sRVO0bSwT

## Fraction

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
        model = "gpt-4o-mini",
        tools = [search_hotels],
        checkpointer=InMemorySaver(),
        middleware=[
            SummarizationMiddleware(
                model="gpt-4o-mini",
                trigger = ("fraction", 0.005),
                keep=("fraction", 0.002)
            )
        ]
    )

config = {"configurable": {"thread_id": "test-1"}}

    # Token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  

In [20]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~141 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='0f72bb63-d70b-4eca-9b42-a4b8b569ed43'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9b01c99668', 'id': 'chatcmpl-Ef0egQb4mI7MdQDr0UuYkQN821vSR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19183-7341-7691-b61f-27513a343e73-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_lBAVHZpHfhVGXh

### Human In the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

    High Stakes operations requiring human approval (e.g. database writes, financial transactions).
    
    Compliance workflows where human oversight is mandatory. 

    Long-running conversations where human feedback guides the agent. 

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by it's ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage 

agent = create_agent(
    model = "gpt-4o",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [32]:

config = {"configurable": {"thread_id" : "test-approve"}}
## Step 1 Request
result = agent.invoke(
    {"messages" : [HumanMessage(content="Send email to john@test.com with the subject 'Hello' and body 'How are you?' ")]},
    config=config
)

In [33]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='294c82e6-567f-4f28-a224-af340908f7c8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 99, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f89046aead', 'id': 'chatcmpl-Ef0pRqdVLUSMxRYB4FaHuv60C91lX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1918d-a1b9-79b0-9845-20fa4c1a382d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recip

In [34]:
# Step 2: Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type" : "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result : {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result : The email has been sent to john@test.com with the subject 'Hello' and the body 'How are you?'.


### Reject

In [35]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage 

agent = create_agent(
    model = "gpt-4o",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [36]:

config = {"configurable": {"thread_id" : "test-reject"}}
## Step 1 Request
result = agent.invoke(
    {"messages" : [HumanMessage(content="Send email to john@test.com with the subject 'Hello' and body 'How are you?' ")]},
    config=config
)

In [37]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type" : "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result : {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result : It looks like the email wasn't sent. If you want me to try again or if there's anything else you'd like to do, just let me know!


In [38]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='2b213ed8-dafe-488b-81a3-21ef624a594d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 99, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f89046aead', 'id': 'chatcmpl-Ef0rP7wA2a6Rdqzhxl98uICeDNRQS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1918f-7da8-7ca3-8a28-03b40cc490bb-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recip

### Editing

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage 

agent = create_agent(
    model = "gpt-4o",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            },
            edit_notice=None,
        )
    ]
)

In [49]:

config = {"configurable": {"thread_id" : "test-edit"}}
## Step 1 Request
result = agent.invoke(
    {"messages" : [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello' ")]},
    config=config
)

In [50]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello' ", additional_kwargs={}, response_metadata={}, id='7ed9faf2-9621-4f77-bd83-9cd7f5b3b29b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 96, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f89046aead', 'id': 'chatcmpl-Ef19FP8pQVCMOxUlDfdl4CK9WXb3X', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a191a0-5f42-7d70-ba92-7ecd7394a78a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wr

In [51]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )
    print("✏️ The email has been successfully sent")


⏸️ Paused! Editing...
✏️ The email has been successfully sent
